<a href="https://colab.research.google.com/github/Decoding-Data-Science/Buildingaiapplicationchallenge2026/blob/main/claude_fable_fallback_simple_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# When Claude Fable 5 Goes Down — Simple Model Fallback Demo

**Goal:** show AI builders how to call a preferred Claude model first, and automatically fall back to another Claude model if the preferred model is unavailable.

This notebook is intentionally simple for a live career talk/workshop.

What you will demonstrate:

1. Make a normal Claude API call.
2. Catch a model availability error.
3. Fall back to a working model.
4. Print which model was actually used.
5. Keep the user experience smooth even when the primary model fails.

## Why this matters

Claude Fable 5 and Claude Mythos 5 created a lot of attention because they were positioned for demanding reasoning and long-horizon agentic work.

But the real production lesson is this:

> Do not build your AI application assuming one model will always be available.

In production, you need a fallback strategy for cases like:

- model unavailable
- wrong model ID
- permission/access issue
- rate limit error
- overloaded API
- timeout or temporary server issue

Official references:

- Anthropic model overview: https://platform.claude.com/docs/en/about-claude/models/overview  
- Anthropic Python SDK: https://platform.claude.com/docs/en/cli-sdks-libraries/sdks/python  
- Anthropic API errors: https://platform.claude.com/docs/en/api/errors  
- Claude status: https://status.claude.com/

In [ ]:
# Install the Anthropic SDK if needed
%pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 923.9/923.9 kB 14.0 MB/s eta 0:00:00


## 1. Setup

Before running this notebook, set your API key as an environment variable:

```bash
export ANTHROPIC_API_KEY="your_api_key_here"
```

On Windows PowerShell:

```powershell
$env:ANTHROPIC_API_KEY="your_api_key_here"
```

In [ ]:
import os
import json
import time
import getpass
from typing import Any, Dict, List, Optional

try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("Enter your ANTHROPIC_API_KEY: ")

os.environ["ANTHROPIC_API_KEY"] = api_key

print("✅ Anthropic API key loaded.")

✅ Anthropic API key loaded.


In [ ]:
import os
from anthropic import (
    Anthropic,
    NotFoundError,
    RateLimitError,
    APIStatusError,
    APIConnectionError,
    AuthenticationError,
    PermissionDeniedError,
)

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise ValueError("Please set ANTHROPIC_API_KEY before running this notebook.")

# max_retries=0 keeps the demo clear.
# The SDK has automatic retries by default, but for teaching we want to show the fallback ourselves.
client = Anthropic(api_key=api_key, max_retries=0)

print("Anthropic client is ready.")

Anthropic client is ready.


## 2. Choose the preferred model and fallback model

For the live demo:

- `MODEL_FABLE` is the preferred model.
- `MODEL_FALLBACK` is the backup model.

If Fable becomes available again and you still want to force the fallback demo, set `MODEL_PREFERRED_FOR_DEMO` to a fake model ID such as `"claude-fable-5-demo-not-found"`.

In [ ]:
MODEL_FABLE = "claude-fable-5"
MODEL_FALLBACK = "claude-opus-4-8"

# Use this for the real demo:
MODEL_PREFERRED_FOR_DEMO = MODEL_FABLE

# If Fable is available again and you want to force the fallback behavior, use this instead:
# MODEL_PREFERRED_FOR_DEMO = "claude-fable-5-demo-not-found"

print("Preferred model:", MODEL_PREFERRED_FOR_DEMO)
print("Fallback model:", MODEL_FALLBACK)

Preferred model: claude-fable-5
Fallback model: claude-opus-4-8


## 3. Prompt for the demo

This prompt connects the technical demo to the current AI news and makes it easy for learners to understand the context.

In [ ]:
basic_prompt = '''
Explain Anthropic Claude Fable 5 and Claude Mythos 5 in simple terms for AI builders.

Launch link:
https://www.anthropic.com/news/claude-fable-5-mythos-5

Include:
1. What it is
2. What it is best for
3. When not to use it
4. One practical example for building AI applications

Keep the answer simple, practical, and useful for developers.
'''

## 4. Helper function: call Claude once

This function only does one job:

> Send a prompt to one model and return the response.

Keeping this separate makes the fallback logic easier to explain.

In [ ]:
def extract_text(message):
    """Extract text blocks from a Claude message response."""
    text_blocks = []

    for block in message.content:
        if getattr(block, "type", None) == "text":
            text_blocks.append(block.text)

    return "\n".join(text_blocks)


def run_claude(model, prompt, max_tokens=900, system=None):
    """Run one Claude Messages API request."""
    request = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "user", "content": prompt}
        ],
    }

    if system:
        request["system"] = system

    message = client.messages.create(**request)

    return {
        "model": model,
        "text": extract_text(message),
        "usage": message.usage,
        "stop_reason": message.stop_reason,
        "request_id": getattr(message, "_request_id", None),
    }


print("run_claude function is ready.")

run_claude function is ready.


## 5. Fallback function

This is the main teaching point.

Simple logic:

1. Try the preferred model.
2. If it fails due to availability/rate/temporary API issues, try the fallback model.
3. If authentication or permission fails, do not retry blindly. Fix the key/access.
4. Print what happened so developers can debug.

In [ ]:
def run_claude_with_fallback(
    preferred_model,
    fallback_model,
    prompt,
    max_tokens=900,
    system=None,
):
    """Try the preferred model first. If it fails, fall back to another model."""

    print(f"Trying preferred model: {preferred_model}")

    try:
        result = run_claude(
            model=preferred_model,
            prompt=prompt,
            max_tokens=max_tokens,
            system=system,
        )

        print(f"Success with preferred model: {preferred_model}")
        result["fallback_used"] = False
        return result

    except (AuthenticationError, PermissionDeniedError) as error:
        print("Authentication or permission error.")
        print("Do not fallback blindly. Fix the API key, billing, or model access.")
        print("Error:", error)
        raise

    except (NotFoundError, RateLimitError, APIConnectionError, APIStatusError) as error:
        status_code = getattr(error, "status_code", None)

        print(f"Preferred model failed: {type(error).__name__}")
        if status_code:
            print(f"Status code: {status_code}")

        print(f"Falling back to: {fallback_model}")

        try:
            result = run_claude(
                model=fallback_model,
                prompt=prompt,
                max_tokens=max_tokens,
                system=system,
            )

            print(f"Success with fallback model: {fallback_model}")
            result["fallback_used"] = True
            result["preferred_model"] = preferred_model
            return result

        except Exception as fallback_error:
            print("Fallback model also failed.")
            print("Error:", fallback_error)
            raise

    except Exception as error:
        print("Unexpected error.")
        print("Error:", error)
        raise


print("run_claude_with_fallback function is ready.")

run_claude_with_fallback function is ready.


## 6. Run the fallback demo

This is the cell you can run live.

Expected teaching moment:

- If the preferred model works, the app uses it.
- If the preferred model is unavailable, the app falls back.
- The final response still reaches the user.

In [ ]:
fallback_demo_result = run_claude_with_fallback(
    preferred_model=MODEL_PREFERRED_FOR_DEMO,
    fallback_model=MODEL_FALLBACK,
    prompt=basic_prompt,
    max_tokens=900,
)

print("\n--- Final Response ---")
print("Model used:", fallback_demo_result["model"])
print("Fallback used:", fallback_demo_result["fallback_used"])
print("Stop reason:", fallback_demo_result["stop_reason"])
print("Request ID:", fallback_demo_result["request_id"])
print("\nResponse text:\n")
print(fallback_demo_result["text"])

Trying preferred model: claude-fable-5
Preferred model failed: NotFoundError
Status code: 404
Falling back to: claude-opus-4-8
Success with fallback model: claude-opus-4-8

--- Final Response ---
Model used: claude-opus-4-8
Fallback used: True
Stop reason: end_turn
Request ID: req_011Cc2s7FxNPMwrfXBFocD2m

Response text:

# Important Note First

I want to be straightforward with you: **"Claude Fable 5" and "Claude Mythos 5" are not real Anthropic products.** I couldn't find any evidence these models exist, and the launch link you provided does not point to a genuine Anthropic announcement.

I can't explain features, use cases, or examples for products that don't exist, because doing so would mean inventing technical details and misleading you.

---

## What's Actually Real (as of my knowledge)

Anthropic's real Claude model families include names like:

- **Claude Opus** – most capable, best for complex reasoning
- **Claude Sonnet** – balanced speed and intelligence
- **Claude Haiku** 

## 7. Simple explanation for the audience

Use this explanation during the session:

> A production AI app should not depend on only one model.  
> The model may be unavailable, rate-limited, restricted, or temporarily overloaded.  
> Instead of showing an error to the user, the app should move to a fallback model and continue working.

This is the foundation of reliable AI application design.

## Optional extension for later

For a more advanced workshop, you can add:

- exponential backoff for 429 errors
- multiple fallback models
- JSON/schema validation
- logging to a database
- user-friendly error messages
- monitoring alerts when fallback usage increases

For this career talk, keep the demo focused:

> Preferred model fails → fallback model works → user still gets an answer.